# Mage AI - Notebook del profesor

Este notebook acompaña la práctica de introducción a **Mage AI** como alternativa ligera y visual a Airflow.

## Objetivos
- Entender qué problema resuelve la orquestación.
- Preparar un ejemplo ETL pequeño con `pandas`.
- Traducir ese ETL a bloques de Mage AI.
- Comparar la experiencia con Airflow.

## Flujo que vamos a construir

Trabajaremos con este esquema:

**Carga de CSV → Transformación con pandas → Exportación de resumen**

En Mage AI esto suele representarse como:
- `Data Loader`
- `Transformer`
- `Data Exporter`

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path('../data/supermarket_sales_small.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## Inspección inicial

Antes de orquestar conviene verificar:
- columnas disponibles,
- tipos de datos,
- calidad básica del dataset.

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## Transformación que usaremos en la práctica

Vamos a:
1. Convertir `Date` a fecha real.
2. Calcular un campo derivado: `ingreso_neto_estimado = Total - Tax 5%`.
3. Agrupar por `Product line`.
4. Obtener:
   - ventas totales,
   - rating medio,
   - número de tickets.

In [ ]:
work = df.copy()
work['Date'] = pd.to_datetime(work['Date'])
work['ingreso_neto_estimado'] = work['Total'] - work['Tax 5%']

resumen = (
    work.groupby('Product line', as_index=False)
    .agg(
        ventas_totales=('Total', 'sum'),
        media_rating=('Rating', 'mean'),
        tickets=('Invoice ID', 'count')
    )
    .sort_values('ventas_totales', ascending=False)
)

resumen

## Exportación local

Esto simula la última etapa del pipeline.

In [ ]:
output_path = Path('../data/resumen_product_line_desde_notebook.csv')
resumen.to_csv(output_path, index=False)
output_path

## Código equivalente para Mage AI

### Bloque 1: Data Loader

In [ ]:
import pandas as pd

def load_data(*args, **kwargs):
    return pd.read_csv('/home/src/data/supermarket_sales_small.csv')

### Bloque 2: Transformer

In [ ]:
def transform(df, *args, **kwargs):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df['ingreso_neto_estimado'] = df['Total'] - df['Tax 5%']
    resumen = (
        df.groupby('Product line', as_index=False)
        .agg(
            ventas_totales=('Total', 'sum'),
            media_rating=('Rating', 'mean'),
            tickets=('Invoice ID', 'count')
        )
        .sort_values('ventas_totales', ascending=False)
    )
    return resumen

### Bloque 3: Data Exporter

In [ ]:
def export_data(df, *args, **kwargs):
    output_path = '/home/src/data/resumen_product_line.csv'
    df.to_csv(output_path, index=False)
    print(f'Fichero exportado en: {output_path}')
    print(df)

## Ejecución simulada fuera de Mage

Así compruebas rápidamente que la lógica del pipeline funciona antes de montarla en la interfaz.

In [ ]:
loaded = pd.read_csv(DATA_PATH)
transformed = loaded.copy()
transformed['Date'] = pd.to_datetime(transformed['Date'])
transformed['ingreso_neto_estimado'] = transformed['Total'] - transformed['Tax 5%']

final_df = (
    transformed.groupby('Product line', as_index=False)
    .agg(
        ventas_totales=('Total', 'sum'),
        media_rating=('Rating', 'mean'),
        tickets=('Invoice ID', 'count')
    )
    .sort_values('ventas_totales', ascending=False)
)

final_df

## Comparativa con Airflow

### Enfoque Mage
- desarrollo visual,
- ejecución por bloques,
- vista previa de datos,
- curva de entrada baja.

### Enfoque Airflow
- definición explícita del DAG,
- mayor formalismo,
- más adecuado cuando la orquestación crece y aparecen entornos, permisos, observabilidad y operación más compleja.

In [ ]:
from textwrap import dedent

airflow_example = dedent("""
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
import pandas as pd

def extract():
    return pd.read_csv('/opt/airflow/data/supermarket_sales_small.csv')

def transform():
    df = extract()
    df['Date'] = pd.to_datetime(df['Date'])
    df['ingreso_neto_estimado'] = df['Total'] - df['Tax 5%']
    resumen = (
        df.groupby('Product line', as_index=False)
        .agg(
            ventas_totales=('Total', 'sum'),
            media_rating=('Rating', 'mean'),
            tickets=('Invoice ID', 'count')
        )
        .sort_values('ventas_totales', ascending=False)
    )
    resumen.to_csv('/opt/airflow/data/resumen_product_line.csv', index=False)

with DAG(
    dag_id='ventas_supermercado',
    start_date=datetime(2026, 1, 1),
    schedule='@daily',
    catchup=False,
) as dag:
    tarea_transform = PythonOperator(
        task_id='transformar_y_exportar',
        python_callable=transform,
    )
""")
print(airflow_example)

## Preguntas para clase
1. ¿Qué parte del proceso pertenece a la orquestación y cuál a la transformación de datos?
2. ¿Qué ventaja ofrece Mage para alumnado que todavía está consolidando Python y pandas?
3. ¿Cuándo empezarían a compensar herramientas más pesadas como Airflow?